In [ ]:
from model_arch import *

In [ ]:
from bpetokenizer import *
tokenizer = load_bpe_tokenizer()

In [ ]:
import pickle, torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEP = tokenizer.encode("####")[0] 
END = tokenizer.encode("</s>")[0]

In [ ]:
class SumDS(Dataset):
    def __init__(self, path):
        self.items = pickle.load(open(path, "rb"))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        ids = self.items[i]["token_ids"]
        p = ids.index(SEP)
        labels = [-100] * (p + 1) + ids[p + 1:]
        return ids, labels

def collate(batch):
    L = max(len(ids) for ids, _ in batch)
    x = torch.zeros(len(batch), L, dtype=torch.long)
    y = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, (ids, labels) in enumerate(batch):
        x[i, :len(ids)] = torch.tensor(ids)
        y[i, :len(labels)] = torch.tensor(labels)
    return x, y

train_dl = DataLoader(SumDS("train_p6.pkl"), batch_size=4, shuffle=True, collate_fn=collate)
dev_dl = DataLoader(SumDS("dev_p6.pkl"), batch_size=4, shuffle=False, collate_fn=collate)

In [ ]:
teacher = DecoderLM(LMConfig(vocab_size=50260)).to(device)
teacher.load_state_dict(torch.load("dec.pt", map_location=device))
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

student = DecoderLM(LMConfig(vocab_size=50260)).to(device)
student.load_state_dict(torch.load("checkpoint.pt", map_location=device))
student.cfg.n_layers = 4
student.blocks = nn.ModuleList([student.blocks[i] for i in [0, 2, 4, 6]])

optimizer = torch.optim.AdamW(student.parameters(), lr=2e-4)

In [ ]:
TEMP = 2.0
ALPHA = 0.5

for epoch in range(6):
    student.train()
    for step, (x, y) in enumerate(train_dl):
        x, y = x.to(device), y.to(device)
        with torch.no_grad():
            t_logits = teacher(x)
        s_logits = student(x)

        mask = y[:, 1:] != -100
        s_shift = s_logits[:, :-1][mask]
        t_shift = t_logits[:, :-1][mask]
        target = y[:, 1:][mask]

        ce_loss = F.cross_entropy(s_shift, target)
        kl_loss = F.kl_div(F.log_softmax(s_shift / TEMP, dim=-1), F.softmax(t_shift / TEMP, dim=-1), reduction="batchmean") * (TEMP ** 2)
        loss = ALPHA * ce_loss + (1 - ALPHA) * kl_loss
                
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()

    student.eval()
    total = 0
    with torch.no_grad():
        for x, y in dev_dl:
            x, y = x.to(device), y.to(device)
            s_logits = student(x)
            loss = F.cross_entropy(s_logits[:, :-1].reshape(-1, s_logits.size(-1)), y[:, 1:].reshape(-1), ignore_index=-100)
            total += loss.item()
    dev_loss = total / len(dev_dl)
    print("dev loss:", dev_loss, "train loss:", loss.item())

    torch.save(student.state_dict(), "p6_full_ft" + str(epoch) + ".pt")